# Formal Layer2R one-case regression

This runtime-only wrapper verifies CUDA, resolves the attached dataset, runs the formal preflight, and executes at most one pending locked case.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import torch

REPOSITORY_MARKERS = ("src", "experiments", "configs")
REPOSITORY_URL = "https://github.com/changxinjiresearch/PCC.git"
APPROVED_COMMIT = "906b39f"
KAGGLE_REPOSITORY_ROOT = Path("/kaggle/working/PCC")


def is_repository_root(path: Path) -> bool:
    return all((path / marker).is_dir() for marker in REPOSITORY_MARKERS)


def run_git(arguments: list[str], *, cwd: Path | None = None) -> subprocess.CompletedProcess[str]:
    try:
        return subprocess.run(
            ["git", *arguments],
            cwd=cwd,
            check=True,
            capture_output=True,
            text=True,
        )
    except subprocess.CalledProcessError as error:
        detail = error.stderr.strip() or error.stdout.strip() or str(error)
        raise RuntimeError(f"Git command failed ({' '.join(arguments)}): {detail}") from error


def verify_kaggle_checkout(repository_root: Path) -> None:
    remote = run_git(["remote", "get-url", "origin"], cwd=repository_root).stdout.strip()
    if remote != REPOSITORY_URL:
        raise RuntimeError(f"Unexpected repository origin at {repository_root}: {remote!r}")
    expected_commit = run_git(["rev-parse", f"{APPROVED_COMMIT}^{{commit}}"], cwd=repository_root).stdout.strip()
    actual_commit = run_git(["rev-parse", "HEAD"], cwd=repository_root).stdout.strip()
    if actual_commit != expected_commit:
        raise RuntimeError(
            f"Repository commit mismatch at {repository_root}: expected {expected_commit}, got {actual_commit}"
        )
    if not is_repository_root(repository_root):
        missing = [marker for marker in REPOSITORY_MARKERS if not (repository_root / marker).is_dir()]
        raise RuntimeError(f"Repository checkout is missing required directories: {', '.join(missing)}")


current_directory = Path.cwd().resolve()
local_roots = [path for path in (current_directory, *current_directory.parents) if is_repository_root(path)]
if local_roots:
    REPOSITORY_ROOT = local_roots[0]
else:
    if KAGGLE_REPOSITORY_ROOT.exists():
        if not KAGGLE_REPOSITORY_ROOT.is_dir():
            raise RuntimeError(f"Kaggle repository path is not a directory: {KAGGLE_REPOSITORY_ROOT}")
    else:
        KAGGLE_REPOSITORY_ROOT.parent.mkdir(parents=True, exist_ok=True)
        try:
            run_git(["clone", REPOSITORY_URL, str(KAGGLE_REPOSITORY_ROOT)])
        except RuntimeError as error:
            raise RuntimeError(
                "Could not clone the PCC repository; ensure Kaggle kernel internet access is enabled. "
                f"{error}"
            ) from error
        run_git(["checkout", "--detach", APPROVED_COMMIT], cwd=KAGGLE_REPOSITORY_ROOT)
    verify_kaggle_checkout(KAGGLE_REPOSITORY_ROOT)
    REPOSITORY_ROOT = KAGGLE_REPOSITORY_ROOT.resolve()

sys.path.insert(0, str(REPOSITORY_ROOT))

from src.pipelines.formal_layer2r import load_formal_config

CONFIG_PATH = REPOSITORY_ROOT / "configs/layer2r_kaggle_one_case.json"
RUNNER_PATH = REPOSITORY_ROOT / "experiments/run_formal_layer2r.py"

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for the formal one-case regression")
print("CUDA device:", torch.cuda.get_device_name(0))

with CONFIG_PATH.open() as stream:
    raw_config = json.load(stream)
if raw_config.get("max_new_cases") != 1:
    raise RuntimeError("One-case wrapper requires max_new_cases == 1")

config = load_formal_config(CONFIG_PATH)
print("Resolved dataset root:", config.raw_root)

command = [sys.executable, str(RUNNER_PATH), "--config", str(CONFIG_PATH)]
preflight = subprocess.run([*command, "--preflight"], check=False)
if preflight.returncode != 0:
    raise RuntimeError(f"Formal one-case preflight failed with exit code {preflight.returncode}")

execution = subprocess.run(command, check=False)
if execution.returncode != 0:
    raise RuntimeError(f"Formal one-case execution failed with exit code {execution.returncode}")
print("Formal Layer2R one-case regression completed; wrapper stopping.")
